# Use Case 2: Rating Bug Severity (the `Score` primitive)

**Recap:** in notebook 1, `Choice` picked one label out of a list (billing / technical / other).

Now we'll learn `Score` — instead of picking a *category*, TypeSafe rates the text on a **scale with described levels**, like "how severe is this bug?" from *cosmetic* to *blocking*.

**The scenario:** we run an engineering team's bug tracker. A user reports a problem, and we want TypeSafe to automatically rate how severe it is, so bad bugs bubble to the top.

### 🧺 Analogy

Imagine 3 baskets on a table, labeled:
- 🧺 Basket 0 = "Not serious" (cosmetic)
- 🧺 Basket 1 = "Kind of serious" (broken, but there's a workaround)
- 🧺 Basket 2 = "Very serious" (totally blocked, no fix)

A person reads the bug report and has to drop it into one basket.

- **Score** = which basket it landed in (or *between* two, if they couldn't fully decide).
- **Confidence** = how steady that person's hand was when dropping it in.

**Example 1 — a typo in the footer ("Copyright 2023" should say 2024):** the person reads it, immediately thinks "that's nothing, doesn't break anything," and drops it straight into Basket 0 without hesitating at all → `score = 0.0`, `confidence = 1.0` (100% sure, no doubt).

**Example 2 — "the export button crashes the settings page in Safari":** the person hesitates — "it's broken, but it only affects Safari and you can still export elsewhere... is that Basket 1 or leaning toward Basket 2?" Their hand hovers between the two → `score = 1.43`, `confidence = 0.35` (a genuinely close call).

Keep this picture in mind — everything below is just how we describe the baskets and the bug report to the computer, and how we read back where it landed.

## Step 1 — Install the SDK

(If you already ran notebook 1 in this same environment, this is likely already installed — running it again is harmless, it'll just say "already satisfied".)

In [2]:
%pip install typesafe-sdk

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Add your API key

Same as last time: paste your real key between the quotes below. Each notebook sets this itself so it can run standalone.

In [ ]:
import os

# 👉 PASTE YOUR TYPESAFE API KEY BELOW, between the quotes
os.environ["TYPESAFE_API_KEY"] = "PASTE_YOUR_API_KEY_HERE"

print("Key saved for this session.")

## Step 3 — Create the client

Same `TypeSafeClient` as before. This time we also import `Score` instead of `Choice`.

In [4]:
from typesafe_sdk import Score, TypeSafeClient

client = TypeSafeClient()
print("Client is ready.")

Client is ready.


## Step 4 — The "state" (the bug report)

Same idea as before: `state` is just the text TypeSafe should look at.

In [5]:
bug_report = "The export button crashes the settings page in Safari."

state = {"document": bug_report}

## Step 5 — Define the question

A `Score` question needs:

- `instructions`: the question to ask.
- `criteria`: an **ordered list** of level descriptions, from lowest to highest severity. Minimum 2 levels, maximum 10.

The levels are numbered starting at 0. With 3 levels like below, the possible score range is 0–2 (and it can land *between* two levels, e.g. 1.4, if it's not fully sure which one fits best).

💡 Tip from the docs: describe each level with a **concrete situation** ("broken feature, workaround exists") rather than a vague word ("medium severity") — it helps TypeSafe judge more consistently.

In [6]:
questions = {
    "bug_severity": Score(
        instructions="How severe is the reported issue?",
        criteria=[
            "Cosmetic; no functional impact",              # level 0
            "Broken feature, but a workaround exists",      # level 1
            "Blocking issue; no workaround available",      # level 2
        ],
    ),
}

## Step 6 — Send it to TypeSafe

Exactly the same method call as notebook 1: `client.system_one(state=..., questions=...)`.

In [7]:
response = client.system_one(
    state=state,
    questions=questions,
)

response

SystemOneResponse(model='jev-1.13.0', usage=Usage(input_tokens=330, output_tokens=20), answers={'bug_severity': ScoreAnswer(type='score', score=1.56, confidence=0.33, legend={0: 'Cosmetic; no functional impact', 1: 'Broken feature, but a workaround exists', 2: 'Blocking issue; no workaround available'}, probabilities={0: 0.0, 1: 0.44, 2: 0.56})})

## Step 7 — Read the answer

For a `Score` question, `response.scores["your_question_name"]` gives you:

- **`score`** — the overall rating (a number, can be a decimal like `1.43` if it's between two levels).
- **`confidence`** — 0 to 1, how sure it is (1.0 = landed squarely on one level, lower = spread across levels/unsure).
- **`probabilities`** — how likely *each* level was.
- **`legend`** — a reminder of what each level number means (handy so you don't have to remember your own `criteria` list).

In [8]:
answer = response.scores["bug_severity"]

print("Score        :", answer.score)
print("Confidence   :", answer.confidence)
print("Probabilities:", answer.probabilities)
print("Legend       :", answer.legend)

Score        : 1.56
Confidence   : 0.33
Probabilities: {0: 0.0, 1: 0.44, 2: 0.56}
Legend       : {0: 'Cosmetic; no functional impact', 1: 'Broken feature, but a workaround exists', 2: 'Blocking issue; no workaround available'}


For this exact bug report, the docs report a score around **1.43** — leaning towards "broken feature, workaround exists" but with some pull towards "blocking", and a fairly low confidence (~0.35) because it's genuinely a borderline case (Safari-only bug — annoying, but not blocking for everyone).

## Step 8 — Try it yourself

Edit `bug_report_2` below — try something clearly minor (a typo) and something clearly severe ("the app deletes all user data on login") and see how the score moves.

In [9]:
# ✏️ Edit this text, then re-run this cell (Shift+Enter)
bug_report_2 = "There's a typo in the footer: 'Copyright 2023' should say 2024."

response_2 = client.system_one(
    state={"document": bug_report_2},
    questions=questions,  # reusing the same question from Step 5
)

answer_2 = response_2.scores["bug_severity"]
print("Score        :", answer_2.score)
print("Confidence   :", answer_2.confidence)
print("Probabilities:", answer_2.probabilities)

Score        : 0.0
Confidence   : 1.0
Probabilities: {0: 1.0, 1: 0.0, 2: 0.0}


## Step 9 — Clean up

In [10]:
client.close()
print("Closed.")

Closed.


## Recap — what did we just learn?

1. `Score(instructions=..., criteria=[level0, level1, ..., levelN])` rates text on an ordered scale instead of picking a category.
2. `criteria` is a **list** (ordered, low → high), not a dictionary like `Choice` used.
3. `response.scores["your_question_name"]` gives you `.score`, `.confidence`, `.probabilities`, and `.legend`.
4. A low `confidence` isn't an error — it's useful information! It tells you "this was a genuinely borderline case", which you can use in your code (e.g. send low-confidence bugs to a human for a second look).

This same pattern works for any "rate this on a scale" use case: urgency, content quality, risk level, lead scoring, etc.

👉 Once this makes sense, let's move on to notebook 3 — the `Noul` primitive (simple yes/no questions).